In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [21]:
df=pd.read_csv("../Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Predicting Salary

In [22]:
df=df.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [23]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [24]:
# Encode Categorical variables
label_encoder_gender=LabelEncoder()
df['Gender']=label_encoder_gender.fit_transform(df['Gender'])

In [25]:
onehot_encoder_geo=OneHotEncoder()
encoder_geo=onehot_encoder_geo.fit_transform(df[['Geography']]).toarray()
encoder_geo

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [26]:
geo_df=pd.DataFrame(encoder_geo,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [27]:
df=pd.concat([df.drop(['Geography'],axis=1),geo_df],axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [28]:
## split the data into features
X=df.drop(["EstimatedSalary"],axis=1)
y=df['EstimatedSalary']

In [29]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## Scale these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [30]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [31]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [37]:
## Build ANN

model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),## HL1 connected with input layer
    Dense(32,activation='relu'),#HL2
    Dense(1) #o/p 
])

c:\Users\dheer\anaconda3\envs\deeplearning\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.MeanAbsoluteError()
### compile(optimizer="adam",  if we provide this , it has its own learning rate so by above step we can declare our learning rate
## compile the model --- for forward and backward propogation

#model.compile(optimizer="adam",loss="binary_crossentropy",metrics=['accuracy'])
model.compile(optimizer=opt,loss=loss,metrics=['mae'])

In [42]:
## setup tensorboard -- to save the logs

log_dir='logs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [44]:
## setup EarlyStopping
early_stopping=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [45]:
## train the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping]
)


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - loss: 81977.2188 - mae: 81977.2188 - val_loss: 51703.7188 - val_mae: 51703.7188
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 50489.6523 - mae: 50489.6523 - val_loss: 50536.2891 - val_mae: 50536.2891
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 49958.2227 - mae: 49958.2227 - val_loss: 50445.3008 - val_mae: 50445.3008
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 49784.9102 - mae: 49784.9102 - val_loss: 50357.8867 - val_mae: 50357.8867
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 49668.6250 - mae: 49668.6250 - val_loss: 50328.1406 - val_mae: 50328.1406
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 49596.3281 - mae: 49596.3281 - val_loss: 50301.5430 - val_mae: 50301.5430
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 49532.9609 - mae: 49532.9609 - val_loss: 50415.7617 - val_mae: 50415.7617
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms

In [46]:
%load_ext tensorboard

In [47]:
%tensorboard --logdir logs/fit

In [48]:
# eval model on the test data

test_loss,test_mae=model.evaluate(X_test,y_test)
print(f"TEst mae :{test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50301.5430 - mae: 50301.5430
TEst mae :50301.54296875
